In [1]:
import importlib
import sys
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, HTML, Markdown

# -------------------- Dark Theme --------------------
display(HTML("""
<style>
    body, .jp-Notebook, .jp-OutputArea-output, .jp-RenderedHTMLCommon {
        background-color: #1e1e1e !important;
        color: #d4d4d4 !important;
    }
    h2, h3, h4 {
        color: #4fc3f7 !important;
        border-bottom: 2px solid #3498db !important;
        padding-bottom: 4px;
    }
    b, strong { color: #f48fb1 !important; }
    .highlight {
        background-color: #2d2d2d !important;
        padding: 10px;
        border-left: 4px solid #3498db;
        margin: 4px 0;
        color: #d4d4d4;
    }
    code {
        background-color: #333 !important;
        color: #ffcc80 !important;
        padding: 2px 4px;
        border-radius: 4px;
    }
    .dataframe {
        background-color: #2d2d2d !important;
        color: #d4d4d4 !important;
    }
</style>
"""))

# -------------------- Verbosity Flags --------------------
SHOW_VERBOSE = True
SHOW_INFO = True
SHOW_CRITICAL = True
SHOW_DEBUG = True

# -------------------- Helper functions for display --------------------
def display_title(title: str):
    display(HTML(f"<h2>{title}</h2>"))

def display_info(message: str):
    display(HTML(f"<div class='highlight'>{message}</div>"))

print("Environment ready. Dark theme applied.")

Environment ready. Dark theme applied.


In [2]:
# =============================================================================
# CELL 2 – LOAD AND INSPECT DATASETS
# =============================================================================

from Get_Go_Emo import get_go
from Get_Isear import get_isr

go_df = get_go()
isear_df = get_isr()

DATASETS = {
    "goEmo": go_df,
    "ISEAR": isear_df,
}

for name, df in DATASETS.items():
    display_title(f"Dataset: {name}")
    display_info(f"Shape: {df.shape}")
    display(df.head(2))
    # We'll rely on the probe's column detection later
    display_info(f"Samples: <b>{len(df):,}</b>")

display_title("Unified ID Scheme")
display_info("""
Each sample is assigned a unique integer ID (0..N-1) matching its row index.
This ID links:
• Input text (original DataFrame index)
• Hidden state vector (row in hidden_states.npy)
• Label (row in labels.npy)
No shuffling occurs, guaranteeing one‑to‑one mapping.
""")

,labels,clean_text
0,[27],my favourite food is anything i didnt have to ...
1,[27],"now if he does off himself, everyone will thin..."


,clean_text,labels
0,during the period of falling in love each time...,1
1,when i was involved in a traffic accident,2


In [3]:
import Probe as probe

GOEMOTIONS_CLASSES = probe.GOEMOTIONS_CLASSES
ISEAR_CLASSES = probe.ISEAR_CLASSES
EXTERNAL_ROOT = Path('/Volumes/Amirali/hidden_states')
EXPERIMENT_ID = 'baseline_v5_001'

# Contracts with auto column detection and lenient provenance
goemotions_contract = probe.DatasetContract(
    target_type='goemotions',
    text_column='auto',
    label_column='auto',
    id_column='auto',
    task_type='multi_label',
    class_order=GOEMOTIONS_CLASSES,
    lenient_provenance=True,      # allow head/tail match
    require_provenance=False,
)

isear_contract = probe.DatasetContract(
    target_type='isear',
    text_column='auto',
    label_column='auto',
    id_column='auto',
    task_type='single_label',
    class_order=ISEAR_CLASSES,
    lenient_provenance=True,
    require_provenance=False,
)

# Probes definition (same as before)
probes = [
    probe.ProbeSpec(name='linear_logistic', type='logistic', complexity='linear',
                    standardize=True, C=1.0, max_iter=3000, selection_metric='macro_f1'),
    probe.ProbeSpec(name='mlp_1_hidden', type='mlp', complexity='1_hidden',
                    standardize=True, hidden_dims=['0.5d'], learning_rate=1e-3,
                    weight_decay=1e-4, epochs=80, batch_size=256, patience=12,
                    selection_metric='macro_f1'),
    probe.ProbeSpec(name='mlp_2_hidden', type='mlp', complexity='2_hidden',
                    standardize=True, hidden_dims=['0.5d', '0.25d'], learning_rate=1e-3,
                    weight_decay=1e-4, epochs=80, batch_size=256, patience=12,
                    selection_metric='macro_f1'),
    probe.ProbeSpec(name='mlp_3_hidden', type='mlp', complexity='3_hidden',
                    standardize=True, hidden_dims=['0.5d', '0.25d', '0.125d'], learning_rate=1e-3,
                    weight_decay=1e-4, epochs=80, batch_size=256, patience=12,
                    selection_metric='macro_f1'),
]
print('Contracts and probes defined (auto columns, lenient provenance).')

Contracts and probes defined (auto columns, lenient provenance).


In [4]:
all_pairs = probe.discover_model_dataset_pairs(EXTERNAL_ROOT, EXPERIMENT_ID)
print(f"Found {len(all_pairs)} model-dataset pairs.")
display(pd.DataFrame(all_pairs))

Found 22 model-dataset pairs.


,model,dataset
0,google-bert/bert-base-uncased,goEmo
1,google-bert/bert-base-uncased,ISEAR
2,distilbert/distilbert-base-uncased,goEmo
3,distilbert/distilbert-base-uncased,ISEAR
4,FacebookAI/roberta-base,goEmo
5,FacebookAI/roberta-base,ISEAR
6,google/electra-small-discriminator,goEmo
7,google/electra-small-discriminator,ISEAR
8,microsoft/deberta-v3-small,goEmo
9,microsoft/deberta-v3-small,ISEAR


In [5]:
dataset_map = {
    'goEmo': (goemotions_contract, go_df),
    'ISEAR': (isear_contract, isear_df),
}

entries = []
for pair in all_pairs:
    model = pair['model']
    dataset = pair['dataset']
    contract, df = dataset_map.get(dataset, (None, None))
    if contract is None:
        continue
    entries.append({
        'model': model,
        'dataset': dataset,
        'contract': contract,
        'dataset_df': df,
    })

print(f"Prepared {len(entries)} matrix entries.")
display(pd.DataFrame(entries)[['model', 'dataset']].head(10))

Prepared 22 matrix entries.


,model,dataset
0,google-bert/bert-base-uncased,goEmo
1,google-bert/bert-base-uncased,ISEAR
2,distilbert/distilbert-base-uncased,goEmo
3,distilbert/distilbert-base-uncased,ISEAR
4,FacebookAI/roberta-base,goEmo
5,FacebookAI/roberta-base,ISEAR
6,google/electra-small-discriminator,goEmo
7,google/electra-small-discriminator,ISEAR
8,microsoft/deberta-v3-small,goEmo
9,microsoft/deberta-v3-small,ISEAR


In [6]:
VERBOSE = 3 # 0 : criticals, 1: insitialisation and target coverage, 2: pre layer and repeat mssg, 3: every detail there is 
# there isn't much information to print for Probes. 
MAX_SAMPLES = None # 2000 # None to use full dataset
REPEATS = 5 # like epochs 

checkpoint_dir = EXTERNAL_ROOT / 'experiments' / EXPERIMENT_ID / 'matrix_checkpoint'

full_results = probe.run_matrix(
    entries,
    external_root=EXTERNAL_ROOT,
    experiment_id=EXPERIMENT_ID,
    probes=probes,
    repeats=REPEATS,
    max_samples=MAX_SAMPLES,
    verbose=VERBOSE,
    checkpoint_dir=checkpoint_dir, 
    shuffled_label_control=False,      # turn off controls to speed up
    shuffled_control_repeats=0,
)


[matrix] 1/22 | google-bert/bert-base-uncased | goEmo | trial 10bae0e6
[probe +    0.00s] ================================================================================================
[probe +    0.00s] INITIALISING UNIFIED HIDDEN-STATE PROBE
[probe +    0.00s] ================================================================================================
[probe +    1.49s] Starting new trial: /Volumes/Amirali/hidden_states/experiments/baseline_v5_001/models/google-bert/bert-base-uncased/datasets/goEmo/analysis/probes/probe_run__google-bert_bert-base-uncased__goEmo__maxfull__rep5__probes=linear_logistic+mlp_1_hidden+mlp_2_hidden+mlp_3_hidden__hash10bae0e67c5e
[probe +    2.02s] Model: google-bert/bert-base-uncased
[probe +    2.02s] Dataset artifact: goEmo
[probe +    2.02s] Hidden-state shape: (54263, 13, 768)
[probe +    2.02s] Task type: multi_label | classes: 28
[probe +    2.02s] Selected layers: 13 | device: cpu
[probe +    2.02s] Alignment: text=verified | labels=unverified


Probing:   0%|          | 0/260 [00:00<?, ?fit/s]

[probe +    2.17s] ================================================================================================
[probe +    2.17s] REPEAT 1/5
[probe +    2.17s] ================================================================================================
[probe +    2.17s] seed=42 | population=54263 | train=43410 | val=5426 | test=5427
[probe +   26.57s] Layer 0 | relative depth=0.000 | geometry silhouette=None
[probe +   27.60s] FIT linear_logistic | layer=0 | complexity=linear | seed=329176201
[probe +   84.90s] TEST Macro-F1=0.9974485017304352 | BalancedAcc=0.9984005268164069 | MCC=0.9974053073858881
[probe +   84.90s] TEST label coverage: positive=28 | both_classes=28 | ROC-AUC=0.9989647761405397 | AP=0.9989049313328958
[probe +   84.90s] FIT mlp_1_hidden | layer=0 | complexity=1_hidden | seed=3129711444
[probe +  130.69s] TEST Macro-F1=0.999042452656733 | BalancedAcc=0.9983923750344756 | MCC=0.9983770913859172
[probe +  130.69s] TEST label coverage: positive=28 | both_class

In [ ]:
print(f"Matrix completed. Full results shape: {full_results.shape}")


In [ ]:
display(full_results.head())

In [ ]:
# Suppose we want metadata for the first row
sample_row = full_results.iloc[0]
metadata_path = sample_row["metadata_path"]
metadata = probe.load_complete_metadata(Path(sample_row["artifact_dir"]))
print(json.dumps(metadata, indent=2))

In [ ]:
if not full_results.empty:
    # Best layer per probe/model/dataset (highest test_macro_f1)
    best_per_probe = full_results.loc[full_results.groupby(["probe", "model", "dataset"])["test_macro_f1"].idxmax()]
    display_title("Best Layer per Probe (Macro-F1)")
    display(best_per_probe[["probe", "model", "dataset", "layer_index", "test_macro_f1", "probe_score"]])

    # Pivot table: model vs best macro-F1 per probe
    pivot_best = best_per_probe.pivot_table(index=["model", "dataset"], columns="probe", values="test_macro_f1")
    display_title("Best Macro-F1 Matrix")
    display(pivot_best.style.background_gradient(cmap='viridis', axis=None))

In [ ]:
output_plots_dir = Path("probe_plots")
output_plots_dir.mkdir(exist_ok=True)

# Use the plotting function from the probe module
probe.plot_full_dashboard(full_results, output_plots_dir)

In [ ]:
# Per model/dataset layer curves
for (model, dataset), group in full_results.groupby(["model", "dataset"]):
    plt.figure(figsize=(12, 6))
    for probe_name in group["probe"].unique():
        sub = group[group["probe"] == probe_name].sort_values("layer_index")
        plt.plot(sub["layer_index"], sub["test_macro_f1"], marker='o', label=probe_name)
    plt.title(f"{model} / {dataset} – Layer-wise Macro-F1")
    plt.xlabel("Layer index")
    plt.ylabel("Macro-F1")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

You are an expert AI research engineer specializing in representation learning, probing methodologies, and emotion recognition. I need your deep analysis and guidance on a critical issue in my project. Below is a comprehensive description of the project, its pipeline, and the specific problem I'm facing. Please provide a sophisticated, actionable response that addresses all points, proposes fixes, and explains the underlying theory and best practices.

Project Overview

I am conducting a large-scale experiment to understand how emotion information is encoded in the hidden states of various pre-trained language models. The goal is to extract hidden states from many models on two emotion datasets and then "probe" these states with simple classifiers to measure how recoverable the emotion labels are at each layer.

Datasets

GoEmotions

54,263 samples
Multi-label: each text can be labeled with one or more of 28 emotions (e.g., admiration, amusement, anger, ... neutral).
Labels are stored as lists of integer IDs.
ISEAR

7,666 samples
Single-label: each text is assigned exactly one of 7 emotions (joy, fear, anger, sadness, disgust, shame, guilt).
Labels are integers 1–7.
Models

I have a registry of 25 pre-trained transformer models of varying sizes (BERT, DistilBERT, RoBERTa, GPT‑2, GPT‑Neo, OPT, SmolLM, Qwen, etc.). All are used only for feature extraction; no fine-tuning is performed.

Pipeline Architecture

Extraction Phase

Deterministic hidden-state extraction pipeline (Extraction6_new).
Models are loaded from a Hugging Face snapshot with pinned revisions.
For each dataset, tokenize texts with max length 512, pass through model, and collect hidden states from all layers (including embeddings).
Hidden states are mean-pooled across tokens (using attention mask).
Stored as memory-mapped .npy arrays with shape (n_samples, n_layers, hidden_size).
For each dataset and model, a directory is created under experiments/{experiment_id}/models/{model_name}/datasets/{dataset_name}/ containing:

data/hidden_states.npy (memmap)
data/completed.npy (bool mask for completed samples)
metadata/extraction.json (full configuration, checksums, provenance)
Sample IDs, text hashes, and integrity hashes.
Probing Phase

Separate module unified_hidden_state_probe_v4_2.py (now v4.3 after enhancements).
For each model-dataset pair, a probing run is performed on the extracted hidden states.
Probes used:

linear_logistic (Logistic Regression with standardization)
mlp_1_hidden (MLP with one hidden layer, width 0.5×input_dim)
mlp_2_hidden (two hidden layers, widths 0.5× and 0.25× input_dim)
mlp_3_hidden (three hidden layers, widths 0.5×, 0.25×, 0.125× input_dim)
Split: 80% train, 10% validation, 10% test, stratified (for single-label) or multilabel-stratified if possible.
Repeats: 2–5 independent repeats with different seeds.
Max samples: either None (full dataset) or a subset (e.g., 2000).
Evaluation metrics:

For single-label: accuracy, balanced accuracy, macro-F1, weighted-F1, MCC, etc.
For multi-label: exact match accuracy, micro-F1, macro-F1, weighted-F1, per-label metrics, ROC-AUC, average precision, etc.
Results are stored per run in a directory structure that now includes deterministic folder names with hyperparameters (e.g., probe_run__Model__Dataset__max2000__rep2__probes=linear_logistic+mlp_1_hidden+...__hash1234567890).
A checkpoint system (matrix_checkpoint) tracks completed trials and allows resumption.
Recent Enhancements

Implemented hyperparameter-aware checkpointing: each trial is uniquely identified by a hash of the full configuration (model, dataset, probes, repeats, max_samples, etc.).
Updated result file naming to include this hash, and created a migration script to index old results.
Added functionality to detect hyperparameters from metadata or folder names.
The Critical Problem

Observation

When running the probing pipeline on GoEmotions with the full dataset (max_samples=None), repeats=5, the test metrics are unrealistically high:

For layer 0, logistic regression achieved:

Macro‑F1 = 0.9974
Balanced accuracy = 0.9984
ROC‑AUC = 0.9989
Average precision = 0.9989
MLP probes also show near‑perfect scores (0.999+).
These scores persist across all 13 layers of BERT‑base, with only minor fluctuations.
This is in stark contrast to earlier runs with max_samples=2000 and repeats=2, where the same model and dataset yielded plausible macro‑F1 values between 0.10 and 0.68. The sudden jump to >0.99 is suspicious and almost certainly indicates data leakage or an evaluation bug, not genuine model capability.

Additional Context

The previous run used shuffled_label_control=False, meaning no randomized label control was performed to detect trivial solutions.
The extraction pipeline is deterministic and should not include label information; however, a bug in split indices or evaluation could cause training/test overlap.
Logistic regression on 28 binary tasks (one per emotion) should not achieve near‑perfect scores on frozen embeddings of a pretrained language model. Typical state‑of‑the‑art for GoEmotions is around 0.5–0.6 macro‑F1.
The Multi‑Label vs Single‑Label Challenge

Beyond the leakage bug, there is a design issue: multi‑label and single‑label datasets require different evaluation and visualization approaches. Currently, results are aggregated without distinguishing the two task types, leading to potential misinterpretation. For example:

Macro‑F1 is computed differently for multi‑label (average over labels) vs single‑label (average over classes).
Chance levels differ: ISEAR has 7 balanced classes (chance macro‑F1 ≈ 0.14), while GoEmotions has 28 imbalanced labels (chance macro‑F1 much lower).
Visualization needs to be tailored: per‑label heatmaps and label cardinality plots for multi‑label; confusion matrices and per‑class curves for single‑label.
Your Task

Please provide a comprehensive analysis covering:

Likely causes of the unrealistically high metrics on full-data GoEmotions. Propose a systematic debugging procedure (e.g., checking split indices, verifying evaluation uses test set only, enabling shuffled-label controls, checking for label leakage in hidden states).
Appropriate handling of multi‑label vs single‑label tasks in the probing pipeline. How should metrics be separated, normalized, and visualized? What common metrics can be used for cross‑dataset comparison?
Is logistic regression a wise choice for multi‑label classification? Explain its role as a linear baseline, its limitations, and how to interpret results relative to non‑linear probes.
Concrete code modification suggestions that can be integrated into the existing pipeline without breaking it. This includes:

Splitting results by task_type for analysis and plotting.
Adding chance‑adjusted metrics (e.g., normalized macro‑F1).
Creating separate visualization functions for single‑label and multi‑label results.
Enhancing the probe scoring system to account for task‑specific difficulty.
A plan to fix the leakage bug while preserving the integrity of already completed runs. How can we verify and correct without re‑running everything?
Please be as detailed and specific as possible, referencing the provided file structure and code patterns where relevant. The response should be a standalone expert analysis that I can implement directly.